In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os
from netCDF4 import Dataset as ncDataset
import datetime
import rasterio
from torch.utils.data import Dataset, DataLoader
import scipy.stats as stats
from tqdm import tqdm
import gc

In [ ]:
import os
import glob
import datetime
import numpy as np
import pandas as pd
import xarray as xr
from scipy import stats

# (0) SPI 계산 함수
def compute_spi(monthly_precip_da: xr.DataArray, scale: int = 3) -> xr.DataArray:
    """
    monthly_precip_da: DataArray(time, lat, lon) 월별 누적강수량
    scale: 누적할 개월 수 (예: 3)
    """
    # 1) rolling sum
    roll = monthly_precip_da.rolling(time=scale, min_periods=scale).sum()

    # 준비
    spi = xr.full_like(roll, np.nan)
    times = roll.time.values
    lats  = roll.lat.values
    lons  = roll.lon.values

    # 그리드별로 순회
    for i, lat in enumerate(lats):
        for j, lon in enumerate(lons):
            series = roll[:, i, j].values
            mask   = np.isfinite(series)
            data   = series[mask]
            if data.size < scale:
                continue

            # 영(0) 값 비율
            p0 = np.sum(data == 0) / data.size
            pos = data[data > 0]
            if pos.size < scale:
                continue

            # 감마 분포 적합 (loc=0 고정)
            alpha, loc, beta = stats.gamma.fit(pos, floc=0)

            # CDF 계산
            cdf = np.empty_like(data, dtype=float)
            for k, v in enumerate(data):
                if v <= 0:
                    cdf[k] = p0 / 2
                else:
                    cdf[k] = p0 + (1 - p0) * stats.gamma.cdf(v, alpha, loc=loc, scale=beta)
            cdf = np.clip(cdf, 1e-6, 1 - 1e-6)

            # 정규 분포 분위수로 변환
            spi_vals = stats.norm.ppf(cdf)

            arr = spi[:, i, j].values
            arr[mask] = spi_vals
            spi[:, i, j] = arr

    spi.name = "SPI"
    spi.attrs["long_name"] = f"SPI-{scale}"
    spi.attrs["units"] = "standardized units"
    return spi


# (1) Co‑kriging 월간 강수량 읽기
def load_monthly_precip(input_folder: str, var_name: str = 'monthly_sum') -> xr.DataArray:
    pattern = os.path.join(input_folder, '*_monthly_sum.nc')
    files   = sorted(glob.glob(pattern))

    da_list = []
    times   = []
    for f in files:
        base     = os.path.basename(f)
        yymm     = base[:6]  # 'YYYYMM'
        year, month = int(yymm[:4]), int(yymm[4:6])
        times.append(np.datetime64(datetime.datetime(year, month, 1)))

        ds = xr.open_dataset(f)
        da = ds[var_name]
        da_list.append(da)
        ds.close()

    precip = xr.concat(da_list, dim='time')
    precip = precip.assign_coords(time=times)
    return precip


# (2) SPI₃ 생성 및 개별 파일 저장
def make_and_save_cokrig_spi3(input_folder: str,
                              output_folder: str,
                              scale_months: int = 3) -> xr.DataArray:
    os.makedirs(output_folder, exist_ok=True)

    # A) 월간 누적강수 읽기
    precip_mon = load_monthly_precip(input_folder, var_name='precipitation')

    # B) SPI 계산
    spi_da = compute_spi(precip_mon, scale=scale_months)

    # C) 파일별 저장
    for t in spi_da.time.values:
        ym    = pd.to_datetime(str(t)).strftime("%Y%m")
        outfn = os.path.join(output_folder, f"{ym}_CoKrig_SPI3.nc")
        spi_da.sel(time=t).rename("SPI").to_dataset().to_netcdf(outfn)
        print(f"✔ 저장: {outfn}")

    return spi_da


# ============================================
# Main
# ============================================
if __name__ == "__main__":
    cokrig_monthly_folder = r"G:\24.11.Graduate\1.CoKriging(Monthly)"
    cokrig_spi_folder     = r"C:\25.01.Drought\SPI(Co-Kriging)"

    cokrig_spi = make_and_save_cokrig_spi3(
        input_folder  = cokrig_monthly_folder,
        output_folder = cokrig_spi_folder,
        scale_months  = 3
    )
    print("=== Co‑kriging SPI₃ 생성 완료 ===")


In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr

def compute_metrics(da_ref: xr.DataArray,
                    da_sim: xr.DataArray,
                    drought_thresh: float = -1.0) -> dict:
    """
    두 SPI DataArray를 같은 좌표(time,lat,lon)로 align 한 뒤
    CC, RMSE, POD, FAR, CSI, HSS 를 계산합니다.
    """
    # 1) 공통 좌표로 정렬 (inner join)
    ref, sim = xr.align(da_ref, da_sim, join='inner')
    
    # 2) flatten & mask NaN
    obs = ref.values.ravel()
    out = sim.values.ravel()
    mask = np.isfinite(obs) & np.isfinite(out)
    obs, out = obs[mask], out[mask]

    # 3) CC & RMSE
    CC   = np.corrcoef(obs, out)[0,1]
    RMSE = np.sqrt(np.mean((out - obs)**2))

    # 4) 가뭄 이벤트 이진화
    obs_d = obs < drought_thresh
    out_d = out < drought_thresh

    TP = np.sum( obs_d &  out_d)
    FP = np.sum(~obs_d &  out_d)
    FN = np.sum( obs_d & ~out_d)
    TN = np.sum(~obs_d & ~out_d)

    POD = TP/(TP+FN) if (TP+FN)>0 else np.nan
    FAR = FP/(TP+FP) if (TP+FP)>0 else np.nan
    CSI = TP/(TP+FP+FN) if (TP+FP+FN)>0 else np.nan
    denom = (TP+FN)*(FN+TN)+(TP+FP)*(FP+TN)
    HSS = (TP*TN - FN*FP)/denom if denom>0 else np.nan

    return {"CC":CC, "RMSE":RMSE,
            "POD":POD, "FAR":FAR, "CSI":CSI, "HSS":HSS}

def load_spi3(folder: str, prefix: str) -> xr.DataArray:
    """
    folder: SPI3 파일이 있는 폴더
    prefix: 'GAN' or 'CoKrig'
    """
    files = sorted(
        os.path.join(folder, fn)
        for fn in os.listdir(folder)
        if fn.endswith(f"{prefix}_SPI3.nc")
    )
    ds = xr.open_mfdataset(files,
                           combine='nested',
                           concat_dim='time')
    return ds["SPI"]

if __name__=="__main__":
    imerg_spi_folder  = r"C:\Yeonji\2025.01.Drought\SPI_netCDF"
    cokrig_spi_folder = r"C:\25.01.Drought\SPI(Co-Kriging)"

    da_imerge = load_spi3(imerg_spi_folder,  "GAN")
    da_cokrig = load_spi3(cokrig_spi_folder, "CoKrig")

    metrics = compute_metrics(da_ref=da_cokrig, da_sim=da_imerge,
                              drought_thresh=-1.0)

    df = pd.DataFrame([metrics], index=["SPI3"])
    print(df)
    df.to_csv(os.path.join(imerg_spi_folder, "SPI3_comparison_metrics.csv"))


연도별 비교 표

In [ ]:
import os
import glob
import xarray as xr
import pandas as pd

In [ ]:
# 1) 분석할 파일들이 들어있는 폴더 경로 (윈도우 경로일 경우 r-string 권장)
folder = r"C:\Yeonji\2025.01.Drought\SPI_netCDF"

# 2) 파일 패턴으로 모든 월별 .nc 파일 찾기
pattern = os.path.join(folder, "*_GAN_SPI3.nc")
file_list = sorted(glob.glob(pattern))

# 3) 월별 통계(공간 평균, 공간 최대, 공간 최소) 기록용 리스트
records = []

for fp in file_list:
    # 4) NetCDF 열기
    ds = xr.open_dataset(fp)
    spi = ds["SPI"]           # 변수 이름이 다르면 여기를 바꿔주세요
    time = pd.to_datetime(ds["time"].values)  # ex. 2023-12-01

    # 5) 공간 통계 계산
    mean_spatial = float(spi.mean(dim=("latitude", "longitude")).values)
    max_spatial  = float(spi.max(dim=("latitude", "longitude")).values)
    min_spatial  = float(spi.min(dim=("latitude", "longitude")).values)

    # 6) 연ㆍ월 정보 추출
    records.append({
        "year":  time.year,
        "month": time.month,
        "mean_SPI": mean_spatial,
        "max_SPI":  max_spatial,
        "min_SPI":  min_spatial
    })

# 7) 판다스 DataFrame 생성
df_monthly = pd.DataFrame(records)

# 8) 연도별 집계: 
#    - 평균 SPI  = 해당 연도의 월별 공간 평균 SPI 의 산술평균
#    - 최대 SPI  = 해당 연도의 월별 공간 최대 SPI 중 최대값
#    - 최소 SPI  = 해당 연도의 월별 공간 최소 SPI 중 최소값
df_annual = (
    df_monthly
    .groupby("year")
    .agg(
        avg_SPI=("mean_SPI", "mean"),
        max_SPI=("max_SPI",   "max"),
        min_SPI=("min_SPI",   "min")
    )
    .reset_index()
)

# 9) 결과 출력
print(df_annual)

# 10) (선택) CSV로 저장하고 싶다면:
# df_annual.to_csv("annual_SPI_stats.csv", index=False)


In [ ]:
# 1) 분석할 파일들이 들어있는 폴더 경로 (윈도우 경로일 경우 r-string 권장)
folder = r"C:\Yeonji\2025.01.Drought\SPI(Co-Kriging)"

# 2) 파일 패턴으로 모든 월별 .nc 파일 찾기
pattern = os.path.join(folder, "*_CoKrig_SPI3.nc")
file_list = sorted(glob.glob(pattern))

# 3) 월별 통계(공간 평균, 공간 최대, 공간 최소) 기록용 리스트
records = []

for fp in file_list:
    # 4) NetCDF 열기
    ds = xr.open_dataset(fp)
    spi = ds["SPI"]           # 변수 이름이 다르면 여기를 바꿔주세요
    time = pd.to_datetime(ds["time"].values)  # ex. 2023-12-01

    # 5) 공간 통계 계산
    mean_spatial = float(spi.mean(dim=("lat", "lon")).values)
    max_spatial  = float(spi.max(dim=("lat", "lon")).values)
    min_spatial  = float(spi.min(dim=("lat", "lon")).values)

    # 6) 연ㆍ월 정보 추출
    records.append({
        "year":  time.year,
        "month": time.month,
        "mean_SPI": mean_spatial,
        "max_SPI":  max_spatial,
        "min_SPI":  min_spatial
    })

# 7) 판다스 DataFrame 생성
df_monthly = pd.DataFrame(records)

# 8) 연도별 집계: 
#    - 평균 SPI  = 해당 연도의 월별 공간 평균 SPI 의 산술평균
#    - 최대 SPI  = 해당 연도의 월별 공간 최대 SPI 중 최대값
#    - 최소 SPI  = 해당 연도의 월별 공간 최소 SPI 중 최소값
df_annual = (
    df_monthly
    .groupby("year")
    .agg(
        avg_SPI=("mean_SPI", "mean"),
        max_SPI=("max_SPI",   "max"),
        min_SPI=("min_SPI",   "min")
    )
    .reset_index()
)

# 9) 결과 출력
print(df_annual)

# 10) (선택) CSV로 저장하고 싶다면:
# df_annual.to_csv("annual_SPI_stats.csv", index=False)


꺾은선 그래프 비교 - 각 파일별 모든 격자의 평균값

In [ ]:
import os
import glob

import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


In [ ]:

# ───────────────────────────────────────────────────────────────
# 1) 파일 리스트 준비
# ───────────────────────────────────────────────────────────────
gan_folder     = r"C:\Yeonji\2025.01.Drought\SPI_netCDF"
gan_pattern    = os.path.join(gan_folder, "*_GAN_SPI3.nc")
gan_files      = sorted(glob.glob(gan_pattern))

obs_folder     = r"C:\Yeonji\2025.01.Drought\SPI(Co-Kriging)"
obs_pattern    = os.path.join(obs_folder, "*_CoKrig_SPI3.nc")
obs_files      = sorted(glob.glob(obs_pattern))


# ───────────────────────────────────────────────────────────────
# 2) 공간평균 시계열 계산 함수
# ───────────────────────────────────────────────────────────────
def load_monthly_mean_spi(file_list, varname="SPI"):
    dates = []
    vals  = []
    for fp in file_list:
        ds   = xr.open_dataset(fp)
        da   = ds[varname]
        # time 차원이 스칼라인 경우에도 .values 로 뽑아 date 변환
        dt   = pd.to_datetime(ds["time"].values)
        mean = float( da.mean(dim=[c for c in da.dims if c not in ("time",)]) )
        dates.append(dt)
        vals.append(mean)
        ds.close()
    return pd.Series(data=vals, index=pd.DatetimeIndex(dates))


gan_series = load_monthly_mean_spi(gan_files, varname="SPI")
obs_series = load_monthly_mean_spi(obs_files, varname="SPI")


# ───────────────────────────────────────────────────────────────
# 3) 시계열 비교 그래프 그리기
# ───────────────────────────────────────────────────────────────
plt.figure(figsize=(10,5))
ax = plt.gca()

# Observed (Co-Kriging)
ax.plot(obs_series.index, obs_series,
        marker="o", linestyle="-", label="Observed",
        color="tab:blue", markersize=4)

# Satellite   (GAN)
ax.plot(gan_series.index, gan_series,
        marker="+", linestyle="-", label="Satellite (GAN)",
        color="tab:red", markersize=6)

# 수평 보조선 at y = -4, -2, 0, 2, 4
for y in [-4, -2, 0, 2, 4]:
    ax.axhline(y, color="gray", linestyle="--", linewidth=0.7)

# 수직 보조선: 짝수년마다
years = range(obs_series.index.year.min(),
              obs_series.index.year.max()+1)
for yr in years:
    if yr % 2 == 0:
        ax.axvline(pd.Timestamp(f"{yr}-01-01"), color="gray",
                   linestyle="--", linewidth=0.7)

# x축 포맷팅: 2년 간격으로 눈금, 'YYYY' 표시
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# 축 레이블 & 한계
ax.set_xlabel("Time (Year)")
ax.set_ylabel("SPI (3-Month)")
ax.set_xlim(obs_series.index.min(), obs_series.index.max())
ax.set_ylim(-4, 4)

# 범례 & 레이아웃
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:

# ───────────────────────────────────────────────────────────────
# 1) 파일 리스트 준비
# ───────────────────────────────────────────────────────────────
gan_folder     = r"C:\Yeonji\2025.01.Drought\SPI3(GAN)"
gan_pattern    = os.path.join(gan_folder, "*_SPI3.nc")
gan_files      = sorted(glob.glob(gan_pattern))

obs_folder     = r"C:\Yeonji\2025.01.Drought\SPI3(Cok)"
obs_pattern    = os.path.join(obs_folder, "*_SPI3.nc")
obs_files      = sorted(glob.glob(obs_pattern))


# ───────────────────────────────────────────────────────────────
# 2) 공간평균 시계열 계산 함수 (수정됨)
# ───────────────────────────────────────────────────────────────
def load_monthly_mean_spi(file_list, varname="SPI3"):
    dates = []
    vals  = []
    
    for fp in file_list:
        try:
            ds = xr.open_dataset(fp)
            da = ds[varname]
            
            # time 값을 가져와서 datetime으로 변환
            time_val = ds["time"].values
            
            # time_val이 이미 datetime 객체인지 확인
            if isinstance(time_val, np.datetime64):
                dt = pd.to_datetime(time_val)
            elif isinstance(time_val, (list, np.ndarray)):
                # 배열이면 첫 번째 요소 사용
                dt = pd.to_datetime(time_val[0])
            else:
                # 스칼라 값인 경우
                dt = pd.to_datetime(time_val)
            
            # 공간 평균 계산
            spatial_dims = [c for c in da.dims if c not in ("time",)]
            mean = float(da.mean(dim=spatial_dims))
            
            dates.append(dt)
            vals.append(mean)
            ds.close()
            
        except Exception as e:
            print(f"Error processing {fp}: {e}")
            continue
    
    # DatetimeIndex 대신 dates 리스트 직접 사용
    return pd.Series(data=vals, index=dates)

# 시계열 데이터 로드
gan_series = load_monthly_mean_spi(gan_files, varname="SPI3")
obs_series = load_monthly_mean_spi(obs_files, varname="SPI3")

# ───────────────────────────────────────────────────────────────
# 3) 시계열 비교 그래프 그리기
# ───────────────────────────────────────────────────────────────
plt.figure(figsize=(10,5))
ax = plt.gca()

# Observed (Co-Kriging)
ax.plot(obs_series.index, obs_series,
        marker="o", linestyle="-", label="Observed",
        color="tab:blue", markersize=4)

# Satellite (GAN)
ax.plot(gan_series.index, gan_series,
        marker="+", linestyle="-", label="Satellite (GAN)",
        color="tab:red", markersize=6)

# 수평 보조선 at y = -4, -2, 0, 2, 4
for y in [-4, -2, 0, 2, 4]:
    ax.axhline(y, color="gray", linestyle="--", linewidth=0.7)

# 수직 보조선: 짝수년마다
years = range(obs_series.index.year.min(),
              obs_series.index.year.max()+1)
for yr in years:
    if yr % 2 == 0:
        ax.axvline(pd.Timestamp(f"{yr}-01-01"), color="gray",
                   linestyle="--", linewidth=0.7)

# x축 포맷팅: 2년 간격으로 눈금, 'YYYY' 표시
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# 축 레이블 & 한계
ax.set_xlabel("Time (Year)")
ax.set_ylabel("SPI (3-Month)")
ax.set_xlim(obs_series.index.min(), obs_series.index.max())
ax.set_ylim(-4, 4)

# 범례 & 레이아웃
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:

# ───────────────────────────────────────────────────────────────
# 1) 파일 리스트 준비
# ───────────────────────────────────────────────────────────────
gan_folder     = r"C:\Yeonji\2025.01.Drought\SPI6(GAN)"
gan_pattern    = os.path.join(gan_folder, "*_SPI6.nc")
gan_files      = sorted(glob.glob(gan_pattern))

obs_folder     = r"C:\Yeonji\2025.01.Drought\SPI6(Cok)"
obs_pattern    = os.path.join(obs_folder, "*_SPI6.nc")
obs_files      = sorted(glob.glob(obs_pattern))
# ───────────────────────────────────────────────────────────────
# 2) 공간평균 시계열 계산 함수 (수정됨)
# ───────────────────────────────────────────────────────────────
def load_monthly_mean_spi(file_list, varname="SPI6"):
    dates = []
    vals  = []
    
    for fp in file_list:
        try:
            ds = xr.open_dataset(fp)
            da = ds[varname]
            
            # time 값을 가져와서 datetime으로 변환
            time_val = ds["time"].values
            
            # time_val이 이미 datetime 객체인지 확인
            if isinstance(time_val, np.datetime64):
                dt = pd.to_datetime(time_val)
            elif isinstance(time_val, (list, np.ndarray)):
                # 배열이면 첫 번째 요소 사용
                dt = pd.to_datetime(time_val[0])
            else:
                # 스칼라 값인 경우
                dt = pd.to_datetime(time_val)
            
            # 공간 평균 계산
            spatial_dims = [c for c in da.dims if c not in ("time",)]
            mean = float(da.mean(dim=spatial_dims))
            
            dates.append(dt)
            vals.append(mean)
            ds.close()
            
        except Exception as e:
            print(f"Error processing {fp}: {e}")
            continue
    
    # DatetimeIndex 대신 dates 리스트 직접 사용
    return pd.Series(data=vals, index=dates)

# 시계열 데이터 로드
gan_series = load_monthly_mean_spi(gan_files, varname="SPI6")
obs_series = load_monthly_mean_spi(obs_files, varname="SPI6")

# ───────────────────────────────────────────────────────────────
# 3) 시계열 비교 그래프 그리기
# ───────────────────────────────────────────────────────────────
plt.figure(figsize=(10,5))
ax = plt.gca()

# Observed (Co-Kriging)
ax.plot(obs_series.index, obs_series,
        marker="o", linestyle="-", label="Observed",
        color="tab:blue", markersize=4)

# Satellite (GAN)
ax.plot(gan_series.index, gan_series,
        marker="+", linestyle="-", label="Satellite (GAN)",
        color="tab:red", markersize=6)

# 수평 보조선 at y = -4, -2, 0, 2, 4
for y in [-4, -2, 0, 2, 4]:
    ax.axhline(y, color="gray", linestyle="--", linewidth=0.7)

# 수직 보조선: 짝수년마다
years = range(obs_series.index.year.min(),
              obs_series.index.year.max()+1)
for yr in years:
    if yr % 2 == 0:
        ax.axvline(pd.Timestamp(f"{yr}-01-01"), color="gray",
                   linestyle="--", linewidth=0.7)

# x축 포맷팅: 2년 간격으로 눈금, 'YYYY' 표시
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# 축 레이블 & 한계
ax.set_xlabel("Time (Year)")
ax.set_ylabel("SPI (6-Month)")
ax.set_xlim(obs_series.index.min(), obs_series.index.max())
ax.set_ylim(-4, 4)

# 범례 & 레이아웃
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:

# ───────────────────────────────────────────────────────────────
# 1) 파일 리스트 준비
# ───────────────────────────────────────────────────────────────
gan_folder     = r"C:\Yeonji\2025.01.Drought\SPI9(GAN)"
gan_pattern    = os.path.join(gan_folder, "*_SPI9.nc")
gan_files      = sorted(glob.glob(gan_pattern))

obs_folder     = r"C:\Yeonji\2025.01.Drought\SPI9(Cok)"
obs_pattern    = os.path.join(obs_folder, "*_SPI9.nc")
obs_files      = sorted(glob.glob(obs_pattern))
# ───────────────────────────────────────────────────────────────
# 2) 공간평균 시계열 계산 함수 (수정됨)
# ───────────────────────────────────────────────────────────────
def load_monthly_mean_spi(file_list, varname="SPI9"):
    dates = []
    vals  = []
    
    for fp in file_list:
        try:
            ds = xr.open_dataset(fp)
            da = ds[varname]
            
            # time 값을 가져와서 datetime으로 변환
            time_val = ds["time"].values
            
            # time_val이 이미 datetime 객체인지 확인
            if isinstance(time_val, np.datetime64):
                dt = pd.to_datetime(time_val)
            elif isinstance(time_val, (list, np.ndarray)):
                # 배열이면 첫 번째 요소 사용
                dt = pd.to_datetime(time_val[0])
            else:
                # 스칼라 값인 경우
                dt = pd.to_datetime(time_val)
            
            # 공간 평균 계산
            spatial_dims = [c for c in da.dims if c not in ("time",)]
            mean = float(da.mean(dim=spatial_dims))
            
            dates.append(dt)
            vals.append(mean)
            ds.close()
            
        except Exception as e:
            print(f"Error processing {fp}: {e}")
            continue
    
    # DatetimeIndex 대신 dates 리스트 직접 사용
    return pd.Series(data=vals, index=dates)

# 시계열 데이터 로드
gan_series = load_monthly_mean_spi(gan_files, varname="SPI9")
obs_series = load_monthly_mean_spi(obs_files, varname="SPI9")

# ───────────────────────────────────────────────────────────────
# 3) 시계열 비교 그래프 그리기
# ───────────────────────────────────────────────────────────────
plt.figure(figsize=(10,5))
ax = plt.gca()

# Observed (Co-Kriging)
ax.plot(obs_series.index, obs_series,
        marker="o", linestyle="-", label="Observed",
        color="tab:blue", markersize=4)

# Satellite (GAN)
ax.plot(gan_series.index, gan_series,
        marker="+", linestyle="-", label="Satellite (GAN)",
        color="tab:red", markersize=6)

# 수평 보조선 at y = -4, -2, 0, 2, 4
for y in [-4, -2, 0, 2, 4]:
    ax.axhline(y, color="gray", linestyle="--", linewidth=0.7)

# 수직 보조선: 짝수년마다
years = range(obs_series.index.year.min(),
              obs_series.index.year.max()+1)
for yr in years:
    if yr % 2 == 0:
        ax.axvline(pd.Timestamp(f"{yr}-01-01"), color="gray",
                   linestyle="--", linewidth=0.7)

# x축 포맷팅: 2년 간격으로 눈금, 'YYYY' 표시
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# 축 레이블 & 한계
ax.set_xlabel("Time (Year)")
ax.set_ylabel("SPI (9-Month)")
ax.set_xlim(obs_series.index.min(), obs_series.index.max())
ax.set_ylim(-4, 4)

# 범례 & 레이아웃
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:

# ───────────────────────────────────────────────────────────────
# 1) 파일 리스트 준비
# ───────────────────────────────────────────────────────────────
gan_folder     = r"C:\Yeonji\2025.01.Drought\SPI12(GAN)"
gan_pattern    = os.path.join(gan_folder, "*_SPI12.nc")
gan_files      = sorted(glob.glob(gan_pattern))

obs_folder     = r"C:\Yeonji\2025.01.Drought\SPI12(Cok)"
obs_pattern    = os.path.join(obs_folder, "*_SPI12.nc")
obs_files      = sorted(glob.glob(obs_pattern))
# ───────────────────────────────────────────────────────────────
# 2) 공간평균 시계열 계산 함수 (수정됨)
# ───────────────────────────────────────────────────────────────
def load_monthly_mean_spi(file_list, varname="SPI12"):
    dates = []
    vals  = []
    
    for fp in file_list:
        try:
            ds = xr.open_dataset(fp)
            da = ds[varname]
            
            # time 값을 가져와서 datetime으로 변환
            time_val = ds["time"].values
            
            # time_val이 이미 datetime 객체인지 확인
            if isinstance(time_val, np.datetime64):
                dt = pd.to_datetime(time_val)
            elif isinstance(time_val, (list, np.ndarray)):
                # 배열이면 첫 번째 요소 사용
                dt = pd.to_datetime(time_val[0])
            else:
                # 스칼라 값인 경우
                dt = pd.to_datetime(time_val)
            
            # 공간 평균 계산
            spatial_dims = [c for c in da.dims if c not in ("time",)]
            mean = float(da.mean(dim=spatial_dims))
            
            dates.append(dt)
            vals.append(mean)
            ds.close()
            
        except Exception as e:
            print(f"Error processing {fp}: {e}")
            continue
    
    # DatetimeIndex 대신 dates 리스트 직접 사용
    return pd.Series(data=vals, index=dates)

# 시계열 데이터 로드
gan_series = load_monthly_mean_spi(gan_files, varname="SPI12")
obs_series = load_monthly_mean_spi(obs_files, varname="SPI12")

# ───────────────────────────────────────────────────────────────
# 3) 시계열 비교 그래프 그리기
# ───────────────────────────────────────────────────────────────
plt.figure(figsize=(10,5))
ax = plt.gca()

# Observed (Co-Kriging)
ax.plot(obs_series.index, obs_series,
        marker="o", linestyle="-", label="Observed",
        color="tab:blue", markersize=4)

# Satellite (GAN)
ax.plot(gan_series.index, gan_series,
        marker="+", linestyle="-", label="Satellite (GAN)",
        color="tab:red", markersize=6)

# 수평 보조선 at y = -4, -2, 0, 2, 4
for y in [-4, -2, 0, 2, 4]:
    ax.axhline(y, color="gray", linestyle="--", linewidth=0.7)

# 수직 보조선: 짝수년마다
years = range(obs_series.index.year.min(),
              obs_series.index.year.max()+1)
for yr in years:
    if yr % 2 == 0:
        ax.axvline(pd.Timestamp(f"{yr}-01-01"), color="gray",
                   linestyle="--", linewidth=0.7)

# x축 포맷팅: 2년 간격으로 눈금, 'YYYY' 표시
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# 축 레이블 & 한계
ax.set_xlabel("Time (Year)")
ax.set_ylabel("SPI (12-Month)")
ax.set_xlim(obs_series.index.min(), obs_series.index.max())
ax.set_ylim(-4, 4)

# 범례 & 레이아웃
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

Scatterplot

In [ ]:
import os
import glob

import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import r2_score

In [ ]:
# ───────────────────────────────────────────────────────────────
# 1) 파일 리스트 준비 (시간순으로 정렬)
# ───────────────────────────────────────────────────────────────
obs_folder  = r"C:\Yeonji\2025.01.Drought\SPI(Co-Kriging)"
gan_folder  = r"C:\Yeonji\2025.01.Drought\SPI_netCDF"

obs_files = sorted(glob.glob(os.path.join(obs_folder, "*_CoKrig_SPI3.nc")))
gan_files = sorted(glob.glob(os.path.join(gan_folder, "*_GAN_SPI3.nc")))

# (여기선 파일 이름 정렬만 동일하게 되었다면, obs_files[i] vs gan_files[i]가 같은 월이어야 합니다)


# ───────────────────────────────────────────────────────────────
# 2) 모든 격자값을 모아 둘 리스트
# ───────────────────────────────────────────────────────────────
obs_all = []
gan_all = []

for fp_obs, fp_gan in zip(obs_files, gan_files):
    ds_o = xr.open_dataset(fp_obs)
    ds_g = xr.open_dataset(fp_gan)

    a = ds_o["SPI"].values.flatten()
    b = ds_g["SPI"].values.flatten()

    # NaN 필터링
    mask = np.isfinite(a) & np.isfinite(b)
    obs_all.append(a[mask])
    gan_all.append(b[mask])

    ds_o.close()
    ds_g.close()

# 리스트 → 1차원 array
x = np.concatenate(obs_all)
y = np.concatenate(gan_all)


# ───────────────────────────────────────────────────────────────
# 3) 회귀분석 & 통계량 계산
# ───────────────────────────────────────────────────────────────
# 직선 계수
slope, intercept = np.polyfit(x, y, 1)
y_pred = slope * x + intercept

# 상관계수 CC
cc = np.corrcoef(x, y)[0,1]
# 결정계수 R²
r2 = r2_score(y, y_pred)


# ───────────────────────────────────────────────────────────────
# 4) 산점도 + 회귀직선 그리기
# ───────────────────────────────────────────────────────────────
plt.figure(figsize=(6,6))
ax = plt.gca()

# 격자점 산점도
ax.scatter(x, y,
           s=1,            # 점 크기
           alpha=0.3,      # 투명도
           color="tab:blue")

# 회귀직선
xx = np.array([-3, 3])
ax.plot(xx, slope*xx + intercept,
        linestyle="--", linewidth=2,
        color="tab:red")

# 축 한계 및 레이블
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_xlabel("Observed SPI (3-Month)")
ax.set_ylabel("Satellite SPI (3-Month)")

# 통계량 텍스트
txt = f"Y = {slope:.3f} X {intercept:+.3f}\nCC = {cc:.3f}\nR² = {r2:.3f}"
ax.text(0.05, 0.95, txt,
        transform=ax.transAxes,
        va="top", ha="left",
        bbox=dict(facecolor="white", alpha=0.7, boxstyle="round"))

plt.tight_layout()
plt.show()

In [ ]:
import os
import glob

import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import r2_score

# ───────────────────────────────────────────────────────────────
# 1) 월별 공간평균 SPI 불러오기 (앞에서 만든 함수 재활용)
# ───────────────────────────────────────────────────────────────
def load_monthly_mean_spi(folder, pattern, varname="SPI"):
    file_list = sorted(glob.glob(os.path.join(folder, pattern)))
    dates = []
    vals  = []
    for fp in file_list:
        ds = xr.open_dataset(fp)
        da = ds[varname]
        # 시간 정보
        dt = pd.to_datetime(ds["time"].values)
        # (lat, lon) 차원을 평균 내면 스칼라가 됨
        mean_val = float( da.mean(dim=[d for d in da.dims if d!="time"]) )
        dates.append(dt)
        vals.append(mean_val)
        ds.close()
    return pd.Series(data=vals, index=pd.DatetimeIndex(dates))

# 코크리깅(Observed) 시리즈
obs_series = load_monthly_mean_spi(
    folder  = r"C:\Yeonji\2025.01.Drought\SPI(Co-Kriging)",
    pattern = "*_CoKrig_SPI3.nc"
)

# GAN(Satellite) 시리즈
gan_series = load_monthly_mean_spi(
    folder  = r"C:\Yeonji\2025.01.Drought\SPI_netCDF",
    pattern = "*_GAN_SPI3.nc"
)

# ───────────────────────────────────────────────────────────────
# 2) 두 시리즈 정렬 후 산점도용 배열 생성
# ───────────────────────────────────────────────────────────────
df = pd.DataFrame({
    "obs": obs_series,
    "gan": gan_series
}).dropna()

x = df["obs"].values
y = df["gan"].values

# 회귀직선 계수 계산
slope, intercept = np.polyfit(x, y, 1)
y_fit = slope * x + intercept

# 상관계수(CC)와 R² 계산
cc = np.corrcoef(x, y)[0,1]
r2 = r2_score(y, y_fit)

# ───────────────────────────────────────────────────────────────
# 3) 산점도 + 회귀직선 그리기
# ───────────────────────────────────────────────────────────────
plt.figure(figsize=(6,6))
ax = plt.gca()

# 산점도
ax.scatter(x, y,
           s=30, color="tab:blue", alpha=0.7,
           label="Monthly mean points")

# 회귀직선
_x = np.array([-3, 3])
ax.plot(_x, slope*_x + intercept,
        linestyle="--", linewidth=2,
        color="tab:red",
        label=f"Y={slope:.3f}X{intercept:+.3f}")

# 축 설정
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_xlabel("Observed SPI(3-Month)", fontsize=12)
ax.set_ylabel("Satellite SPI(3-Month)", fontsize=12)

# 텍스트 박스에 통계량 추가
textstr = (
    f"CC = {cc:.3f}\n"
    f"R² = {r2:.3f}"
)
# axes 좌상단에 박스
ax.text(0.05, 0.95, textstr,
        transform=ax.transAxes,
        fontsize=11, verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7)
)

ax.grid(False)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:
import os
import glob
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import r2_score

# ─── (省略: load_monthly_mean_spi 정의 등은 그대로) ──────────────────────
# 1) obs_series, gan_series 로딩 (생략)
# 2) df 생성 (생략)
x = df["obs"].values
y = df["gan"].values

# ──────────────────────────────────────────────────────────────────────────
#    intercept=0 으로 회귀계수(slope) 계산 (원점을 지나는 직선)
# ──────────────────────────────────────────────────────────────────────────
slope = np.sum(x * y) / np.sum(x * x)  # 원점을 지나는 회귀선의 기울기 계산
y_fit = slope * x  # 예측값 (intercept = 0)

# 상관계수(CC)와 R² 계산
cc = np.corrcoef(x, y)[0, 1]
r2 = r2_score(y, y_fit)  # 원점을 지나는 회귀선에 대한 R²

# ──────────────────────────────────────────────────────────────────────────
# 3) 산점도 + 원점회귀선 그리기
# ──────────────────────────────────────────────────────────────────────────
plt.figure(figsize=(6, 6))
ax = plt.gca()

# 산점도
ax.scatter(x, y,
           s=30, color="tab:blue", alpha=0.7,
           label="Monthly mean points")

# 원점회귀선 - 확실히 (0,0)을 지나도록 그리기
_x = np.array([-3, 0, 3])  # 0 포인트 추가하여 원점을 명시적으로 포함
ax.plot(_x, slope * _x,
        linestyle="--", linewidth=2,
        color="tab:red",
        label=f"Y={slope:.3f}·X")

# 1:1 참조선 추가 (선택사항)
ax.plot([-3, 3], [-3, 3], 
        linestyle="-", linewidth=1, 
        color="gray", alpha=0.5,
        label="1:1 line")

# 원점 강조 (선택사항)
ax.scatter([0], [0], s=50, color="black", marker="+", zorder=10)

# 축 설정
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_xlabel("Observed SPI(3-Month)", fontsize=12)
ax.set_ylabel("Satellite SPI(3-Month)", fontsize=12)
ax.axhline(y=0, color='lightgray', linestyle='-', linewidth=0.5)  # x축 추가
ax.axvline(x=0, color='lightgray', linestyle='-', linewidth=0.5)  # y축 추가

# 통계량 박스
textstr = (
    f"CC = {cc:.3f}\n"
    f"R² = {r2:.3f}\n"
    f"Slope = {slope:.3f}"
)
ax.text(0.05, 0.95, textstr,
        transform=ax.transAxes,
        fontsize=11, verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

ax.grid(False)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("origin_regression_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import os
import glob

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

# 1) 파일 리스트 준비
obs_folder = r"C:\Yeonji\2025.01.Drought\SPI(Co-Kriging)"
gan_folder = r"C:\Yeonji\2025.01.Drought\SPI_netCDF"

obs_files = sorted(glob.glob(os.path.join(obs_folder, "*_CoKrig_SPI3.nc")))
gan_files = sorted(glob.glob(os.path.join(gan_folder, "*_GAN_SPI3.nc")))

# 2) 모든 격자값 수집
obs_all = []
gan_all = []

for fo, fg in zip(obs_files, gan_files):
    ds_o = xr.open_dataset(fo)
    ds_g = xr.open_dataset(fg)
    a = ds_o["SPI"].values.flatten()
    b = ds_g["SPI"].values.flatten()
    mask = np.isfinite(a) & np.isfinite(b)
    obs_all.append(a[mask])
    gan_all.append(b[mask])
    ds_o.close(); ds_g.close()

x = np.concatenate(obs_all)  # Observed
y = np.concatenate(gan_all)  # GAN

# 3) 범주 정의 및 색상 매핑
bins   = [-np.inf, -2, -1.5, -1, 1, 1.5, 2, np.inf]
labels = ["극심건조", "매우건조", "약간건조", "정상", "약간습함", "매우습함", "극심습함"]
cats   = pd.cut(x, bins=bins, labels=labels)

color_map = {
    "극심건조": "#d73027",
    "매우건조": "#fc8d59",
    "약간건조": "#fee090",
    "정상":     "#ffffbf",
    "약간습함": "#91bfdb",
    "매우습함": "#4575b4",
    "극심습함": "#313695",
}

# 4) 회귀분석
slope, intercept = np.polyfit(x, y, 1)
y_pred = slope * x + intercept
cc  = np.corrcoef(x, y)[0,1]
r2  = r2_score(y, y_pred)

# 5) 산점도 그리기
plt.figure(figsize=(6,6))
ax = plt.gca()

for cat in labels:
    mask = (cats == cat)
    ax.scatter(
        x[mask], y[mask],
        s=2, alpha=0.5,
        color=color_map[cat],
        label=cat
    )

# 회귀직선
xx = np.array([-3,3])
ax.plot(xx, slope*xx+intercept,
        linestyle="--", color="k", linewidth=1.5)

# 통계량 텍스트
txt = f"Y={slope:.3f}X{intercept:+.3f}\nCC={cc:.3f}  R²={r2:.3f}"
ax.text(0.05, 0.95, txt,
        transform=ax.transAxes, va="top", bbox=dict(facecolor="white", alpha=0.7))

# 축, 범례
ax.set_xlim(-3,3); ax.set_ylim(-3,3)
ax.set_xlabel("Observed SPI(3-Month)")
ax.set_ylabel("Satellite SPI(3-Month)")
ax.legend(title="습·건 상태", loc="lower right", fontsize=8, framealpha=0.8)

plt.tight_layout()
plt.show()


In [ ]:
# 1) SPI3 파일이 들어있는 폴더 & 패턴 지정
folder  = r"C:\Yeonji\2025.01.Drought\SPI_netCDF"
pattern = "*_GAN_SPI3.nc"    # 관측은 "*_CoKrig_SPI3.nc" 로 바꿔 쓰세요
files   = sorted(glob.glob(os.path.join(folder, pattern)))

# 2) 모든 격자값(flatten) 한 번에 모으기
all_vals = []
for fp in files:
    ds = xr.open_dataset(fp)
    a  = ds["SPI"].values.flatten()
    all_vals.append(a)
    ds.close()
arr = np.concatenate(all_vals)
# NaN 제거
arr = arr[np.isfinite(arr)]

# 3) 가뭄 등급 경계 & 라벨 정의
#    bins = [-inf, -2.0, -1.5, -1.0, 0.99, +inf]
#    labels = ['Extreme Drought','Severe Drought','Moderate Drought','Slight Drought','No Drought']
bins   = [-np.inf, -2.0, -1.5, -1.0, 0.99, np.inf]
labels = ['Extreme Drought','Severe Drought','Moderate Drought','Slight Drought','No Drought']

# 4) pandas.cut 으로 등급 분류
cats = pd.cut(arr, bins=bins, labels=labels, right=False)

# 5) 갯수(count)와 비율(percentage %) 계산
counts    = cats.value_counts().reindex(labels)            # 순서대로
percent   = (counts / counts.sum() * 100).round(2)
summary   = pd.DataFrame({
    'Total Count': counts,
    'Percentage (%)': percent
})
# 6) '3-Month SPI' 라는 인덱스를 MultiIndex 로 넣어주면
summary = pd.concat({'3-Month SPI': summary}, names=['Period'])

print(summary)


In [ ]:
# 1) SPI3 파일이 들어있는 폴더 & 패턴 지정
folder  = r"C:\Yeonji\2025.01.Drought\SPI(Co-Kriging)"
pattern = "*_CoKrig_SPI3.nc"    
files   = sorted(glob.glob(os.path.join(folder, pattern)))

# 2) 모든 격자값(flatten) 한 번에 모으기
all_vals = []
for fp in files:
    ds = xr.open_dataset(fp)
    a  = ds["SPI"].values.flatten()
    all_vals.append(a)
    ds.close()
arr = np.concatenate(all_vals)
# NaN 제거
arr = arr[np.isfinite(arr)]

# 3) 가뭄 등급 경계 & 라벨 정의
#    bins = [-inf, -2.0, -1.5, -1.0, 0.99, +inf]
#    labels = ['Extreme Drought','Severe Drought','Moderate Drought','Slight Drought','No Drought']
bins   = [-np.inf, -2.0, -1.5, -1.0, 0.99, np.inf]
labels = ['Extreme Drought','Severe Drought','Moderate Drought','Slight Drought','No Drought']

# 4) pandas.cut 으로 등급 분류
cats = pd.cut(arr, bins=bins, labels=labels, right=False)

# 5) 갯수(count)와 비율(percentage %) 계산
counts    = cats.value_counts().reindex(labels)            # 순서대로
percent   = (counts / counts.sum() * 100).round(2)
summary   = pd.DataFrame({
    'Total Count': counts,
    'Percentage (%)': percent
})
# 6) '3-Month SPI' 라는 인덱스를 MultiIndex 로 넣어주면
summary = pd.concat({'3-Month SPI': summary}, names=['Period'])

print(summary)


In [ ]:
import glob
import os
import geopandas as gpd
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Point

# ───────────────────────────────────────────────────────────────
# 1) Load watershed boundary shapefile
# ───────────────────────────────────────────────────────────────
wshed_fp = r'G:\24.11.Graduate/유역경계(grs80).shp'
gdf = gpd.read_file(wshed_fp)

# Assuming 'GUBUN' column contains watershed names
regions = gdf['GUBUN'].unique().tolist()

# Create a dictionary to map Korean names to English for display only
region_display_map = {
    '동부수역': 'East',
    '서부수역': 'West', 
    '남부수역': 'South',
    '북부수역': 'North'
}

# Convert to WGS84 if needed
if gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

# ───────────────────────────────────────────────────────────────
# 2) Function to load and analyze SPI data
# ───────────────────────────────────────────────────────────────
def analyze_spi_period(spi_period, data_type, regions, gdf):
    """Analyze drought ratio for a specific SPI period and data type"""
    
    # Set folder path based on data type
    if data_type == 'GAN':
        spi_folder = rf"C:\Yeonji\2025.01.Drought\SPI{spi_period}(GAN)"
    else:  # Cokriging
        spi_folder = rf"C:\Yeonji\2025.01.Drought\SPI{spi_period}(Cok)"
    
    spi_files = sorted(glob.glob(os.path.join(spi_folder, f"*_SPI{spi_period}.nc")))
    
    if len(spi_files) == 0:
        print(f"No files found for SPI{spi_period} ({data_type})")
        return None
    
    print(f"\n=== Processing SPI{spi_period} ({data_type}) ===")
    print(f"Found {len(spi_files)} files")
    
    # Load data
    try:
        spi = xr.open_mfdataset(
            spi_files,
            combine='nested',
            concat_dim='time',
            data_vars='all',
            coords='all',
            compat='override'
        )[f'SPI{spi_period}']
    except:
        # Alternative loading method
        spi_list = []
        for file in spi_files:
            ds = xr.open_dataset(file)
            spi_list.append(ds[f'SPI{spi_period}'])
            ds.close()
        spi = xr.concat(spi_list, dim='time')
    
    # Get coordinates
    if 'latitude' in spi.coords:
        lat = spi['latitude'].values
        lon = spi['longitude'].values
    elif 'lat' in spi.coords:
        lat = spi['lat'].values
        lon = spi['lon'].values
    else:
        lat = spi['y'].values
        lon = spi['x'].values
    
    nlat, nlon = len(lat), len(lon)
    lon_2d, lat_2d = np.meshgrid(lon, lat)
    
    # Generate masks for each watershed
    region_masks = {}
    for region in regions:
        region_polys = gdf[gdf['GUBUN'] == region]
        union_poly = region_polys.unary_union
        
        mask = np.zeros((nlat, nlon), dtype=bool)
        minx, miny, maxx, maxy = union_poly.bounds
        
        for i in range(nlat):
            for j in range(nlon):
                if miny <= lat_2d[i,j] <= maxy and minx <= lon_2d[i,j] <= maxx:
                    pt = Point(lon_2d[i,j], lat_2d[i,j])
                    if union_poly.contains(pt):
                        mask[i, j] = True
        
        region_masks[region] = mask
    
    # Calculate drought event ratios
    drought = (spi <= -1.0)
    
    ratios = {}
    for region, mask in region_masks.items():
        # Convert mask to DataArray
        mask_da = xr.DataArray(mask, dims=['lat', 'lon'])
        
        # Match dimensions
        if 'latitude' in spi.dims:
            mask_da = mask_da.rename({'lat': 'latitude', 'lon': 'longitude'})
        elif 'y' in spi.dims:
            mask_da = mask_da.rename({'lat': 'y', 'lon': 'x'})
        
        # Apply mask
        spi_region = spi.where(mask_da)
        drought_region = drought.where(mask_da)
        
        # Calculate ratio
        valid_count = (~np.isnan(spi_region)).sum().values
        drought_count = drought_region.sum().values
        
        if valid_count > 0:
            ratios[region] = drought_count / valid_count
        else:
            ratios[region] = np.nan
    
    return ratios

# ───────────────────────────────────────────────────────────────
# 3) Analyze multiple SPI periods for both GAN and Cokriging
# ───────────────────────────────────────────────────────────────
spi_periods = [3, 6, 9, 12]
data_types = ['GAN', 'Cok']
all_results = {}

for period in spi_periods:
    for data_type in data_types:
        ratios = analyze_spi_period(period, data_type, regions, gdf)
        if ratios is not None:
            all_results[f'SPI{period}_{data_type}'] = ratios

# ───────────────────────────────────────────────────────────────
# 4) Visualization - GAN vs Cokriging comparison
# ───────────────────────────────────────────────────────────────

# Get English names for display
regions_eng = [region_display_map.get(r, r) for r in regions]

# 4.1) Comparison for each SPI period
for period in spi_periods:
    gan_key = f'SPI{period}_GAN'
    cok_key = f'SPI{period}_Cok'
    
    if gan_key in all_results or cok_key in all_results:
        fig, ax = plt.subplots(figsize=(10, 6))
        
        x = np.arange(len(regions))
        width = 0.35
        
        # GAN data
        if gan_key in all_results:
            gan_values = [all_results[gan_key].get(r, np.nan) for r in regions]
            bars1 = ax.bar(x - width/2, gan_values, width, label='GAN',
                          color='steelblue', edgecolor='black', linewidth=0.5)
            
            # Add value labels
            for bar, value in zip(bars1, gan_values):
                if not np.isnan(value):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                           f'{value:.1%}', ha='center', va='bottom', fontsize=10)
        
        # Cokriging data
        if cok_key in all_results:
            cok_values = [all_results[cok_key].get(r, np.nan) for r in regions]
            bars2 = ax.bar(x + width/2, cok_values, width, label='Cokriging',
                          color='lightsteelblue', edgecolor='black', linewidth=0.5)
            
            # Add value labels
            for bar, value in zip(bars2, cok_values):
                if not np.isnan(value):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                           f'{value:.1%}', ha='center', va='bottom', fontsize=10)
        
        ax.set_xlabel('Watershed', fontsize=14)
        ax.set_ylabel('Averaged Drought Area (%)', fontsize=14)
        #ax.set_title(f'SPI{period} - GAN vs Cokriging Comparison', fontsize=16)
        ax.set_xticks(x)
        ax.set_xticklabels(regions_eng, rotation=0, fontsize=12)
        ax.legend(fontsize=12)
        ax.grid(True, alpha=0.3, axis='y')
        ax.set_ylim(0, 0.4)  # Adjust based on your data
        ax.tick_params(direction='in', labelsize=12)
        
        plt.tight_layout()
        plt.show()

# 4.2) All SPI periods combined comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, period in enumerate(spi_periods):
    ax = axes[idx]
    gan_key = f'SPI{period}_GAN'
    cok_key = f'SPI{period}_Cok'
    
    x = np.arange(len(regions))
    width = 0.35
    
    # GAN data
    if gan_key in all_results:
        gan_values = [all_results[gan_key].get(r, np.nan) for r in regions]
        bars1 = ax.bar(x - width/2, gan_values, width, label='GAN',
                      color='steelblue', edgecolor='black', linewidth=0.5)
    
    # Cokriging data
    if cok_key in all_results:
        cok_values = [all_results[cok_key].get(r, np.nan) for r in regions]
        bars2 = ax.bar(x + width/2, cok_values, width, label='Cokriging',
                      color='lightsteelblue', edgecolor='black', linewidth=0.5)
    
    ax.set_xlabel('Watershed', fontsize=14)
    ax.set_ylabel('Averaged Drought Area (%)', fontsize=14)
    #ax.set_title(f'SPI{period}', fontsize=16)
    ax.set_xticks(x)
    ax.set_xticklabels(regions_eng, rotation=0, fontsize=12)
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 0.4)  # Adjust based on your data
    ax.tick_params(direction='in', labelsize=12)

plt.suptitle('Comparison of Drought Ratios - GAN vs Cokriging', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

# 4.3) Summary statistics comparison
print("\n=== Summary Results - GAN vs Cokriging ===")
for period in spi_periods:
    gan_key = f'SPI{period}_GAN'
    cok_key = f'SPI{period}_Cok'
    
    print(f"\nSPI{period}:")
    print("  GAN:")
    if gan_key in all_results:
        for region, eng_name in zip(regions, regions_eng):
            ratio = all_results[gan_key].get(region, np.nan)
            if not np.isnan(ratio):
                print(f"    {eng_name}: {ratio:.2%}")
            else:
                print(f"    {eng_name}: No data")
        
        # Calculate average
        valid_ratios = [r for r in all_results[gan_key].values() if not np.isnan(r)]
        if valid_ratios:
            avg_ratio = np.mean(valid_ratios)
            print(f"    Average: {avg_ratio:.2%}")
    
    print("  Cokriging:")
    if cok_key in all_results:
        for region, eng_name in zip(regions, regions_eng):
            ratio = all_results[cok_key].get(region, np.nan)
            if not np.isnan(ratio):
                print(f"    {eng_name}: {ratio:.2%}")
            else:
                print(f"    {eng_name}: No data")
        
        # Calculate average
        valid_ratios = [r for r in all_results[cok_key].values() if not np.isnan(r)]
        if valid_ratios:
            avg_ratio = np.mean(valid_ratios)
            print(f"    Average: {avg_ratio:.2%}")

# 4.4) Create a single comprehensive comparison chart
fig, ax = plt.subplots(figsize=(14, 8))

# Prepare data for grouped bar chart
bar_width = 0.15
positions = np.arange(len(regions))
colors_gan = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']  # Blue, Orange, Green, Red
colors_cok = ['#aec7e8', '#ffbb78', '#98df8a', '#ff9896']  # Lighter versions

for i, period in enumerate(spi_periods):
    gan_key = f'SPI{period}_GAN'
    cok_key = f'SPI{period}_Cok'
    
    # Position offset for each SPI period
    offset = (i - 1.5) * bar_width
    
    # GAN bars
    if gan_key in all_results:
        gan_values = [all_results[gan_key].get(r, 0) * 100 for r in regions]  # Convert to percentage
        ax.bar(positions + offset - bar_width/2, gan_values, bar_width,
               label=f'SPI{period}-GAN', color=colors_gan[i], edgecolor='black', linewidth=0.5)
    
    # Cokriging bars  
    if cok_key in all_results:
        cok_values = [all_results[cok_key].get(r, 0) * 100 for r in regions]  # Convert to percentage
        ax.bar(positions + offset + bar_width/2, cok_values, bar_width,
               label=f'SPI{period}-Cok', color=colors_cok[i], edgecolor='black', linewidth=0.5)

ax.set_xlabel('Watershed', fontsize=16)
ax.set_ylabel('Averaged Drought Area (%)', fontsize=16)
#ax.set_title('Comprehensive Drought Analysis - GAN vs Cokriging for All SPI Periods', fontsize=18)
ax.set_xticks(positions)
ax.set_xticklabels(regions_eng, rotation=0, fontsize=14)
ax.legend(ncol=4, loc='upper center', bbox_to_anchor=(0.5, 1.15), fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 40)  # Adjust based on your data
ax.tick_params(direction='in', labelsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────
# 0) Set up output directory for saved figures
# ───────────────────────────────────────────────────────────────
# Create a directory to save figures if it doesn't exist
output_dir = r'C:\Yeonji\2025.01.Drought\Figures'
os.makedirs(output_dir, exist_ok=True)
print(f"Figures will be saved to: {output_dir}")

# ───────────────────────────────────────────────────────────────
# 1) Load watershed boundary shapefile
# ───────────────────────────────────────────────────────────────
wshed_fp = r'G:\24.11.Graduate/유역경계(grs80).shp'
gdf = gpd.read_file(wshed_fp)

# Assuming 'GUBUN' column contains watershed names
regions = gdf['GUBUN'].unique().tolist()

# Create a dictionary to map Korean names to English for display only
region_display_map = {
    '동부수역': 'East',
    '서부수역': 'West', 
    '남부수역': 'South',
    '북부수역': 'North'
}

# Convert to WGS84 if needed
if gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

# ───────────────────────────────────────────────────────────────
# 2) Function to load and analyze SPI data
# ───────────────────────────────────────────────────────────────
def analyze_spi_period(spi_period, data_type, regions, gdf):
    """Analyze drought ratio for a specific SPI period and data type"""
    
    # Set folder path based on data type
    if data_type == 'GAN':
        spi_folder = rf"C:\Yeonji\2025.01.Drought\SPI{spi_period}(GAN)"
    else:  # Cokriging
        spi_folder = rf"C:\Yeonji\2025.01.Drought\SPI{spi_period}(Cok)"
    
    spi_files = sorted(glob.glob(os.path.join(spi_folder, f"*_SPI{spi_period}.nc")))
    
    if len(spi_files) == 0:
        print(f"No files found for SPI{spi_period} ({data_type})")
        return None
    
    print(f"\n=== Processing SPI{spi_period} ({data_type}) ===")
    print(f"Found {len(spi_files)} files")
    
    # Load data
    try:
        spi = xr.open_mfdataset(
            spi_files,
            combine='nested',
            concat_dim='time',
            data_vars='all',
            coords='all',
            compat='override'
        )[f'SPI{spi_period}']
    except:
        # Alternative loading method
        spi_list = []
        for file in spi_files:
            ds = xr.open_dataset(file)
            spi_list.append(ds[f'SPI{spi_period}'])
            ds.close()
        spi = xr.concat(spi_list, dim='time')
    
    # Get coordinates
    if 'latitude' in spi.coords:
        lat = spi['latitude'].values
        lon = spi['longitude'].values
    elif 'lat' in spi.coords:
        lat = spi['lat'].values
        lon = spi['lon'].values
    else:
        lat = spi['y'].values
        lon = spi['x'].values
    
    nlat, nlon = len(lat), len(lon)
    lon_2d, lat_2d = np.meshgrid(lon, lat)
    
    # Generate masks for each watershed
    region_masks = {}
    for region in regions:
        region_polys = gdf[gdf['GUBUN'] == region]
        union_poly = region_polys.unary_union
        
        mask = np.zeros((nlat, nlon), dtype=bool)
        minx, miny, maxx, maxy = union_poly.bounds
        
        for i in range(nlat):
            for j in range(nlon):
                if miny <= lat_2d[i,j] <= maxy and minx <= lon_2d[i,j] <= maxx:
                    pt = Point(lon_2d[i,j], lat_2d[i,j])
                    if union_poly.contains(pt):
                        mask[i, j] = True
        
        region_masks[region] = mask
    
    # Calculate drought event ratios
    drought = (spi <= -1.0)
    
    ratios = {}
    for region, mask in region_masks.items():
        # Convert mask to DataArray
        mask_da = xr.DataArray(mask, dims=['lat', 'lon'])
        
        # Match dimensions
        if 'latitude' in spi.dims:
            mask_da = mask_da.rename({'lat': 'latitude', 'lon': 'longitude'})
        elif 'y' in spi.dims:
            mask_da = mask_da.rename({'lat': 'y', 'lon': 'x'})
        
        # Apply mask
        spi_region = spi.where(mask_da)
        drought_region = drought.where(mask_da)
        
        # Calculate ratio
        valid_count = (~np.isnan(spi_region)).sum().values
        drought_count = drought_region.sum().values
        
        if valid_count > 0:
            ratios[region] = drought_count / valid_count
        else:
            ratios[region] = np.nan
    
    return ratios

# ───────────────────────────────────────────────────────────────
# 3) Analyze multiple SPI periods for both GAN and Cokriging
# ───────────────────────────────────────────────────────────────
spi_periods = [3, 6, 9, 12]
data_types = ['GAN', 'Cok']
all_results = {}

for period in spi_periods:
    for data_type in data_types:
        ratios = analyze_spi_period(period, data_type, regions, gdf)
        if ratios is not None:
            all_results[f'SPI{period}_{data_type}'] = ratios

# ───────────────────────────────────────────────────────────────
# 4) Visualization - GAN vs Cokriging comparison
# ───────────────────────────────────────────────────────────────

# Get English names for display
regions_eng = [region_display_map.get(r, r) for r in regions]

# 4.1) Comparison for each SPI period
for period in spi_periods:
    gan_key = f'SPI{period}_GAN'
    cok_key = f'SPI{period}_Cok'
    
    if gan_key in all_results or cok_key in all_results:
        fig, ax = plt.subplots(figsize=(10, 6))
        
        x = np.arange(len(regions))
        width = 0.35
        
        # GAN data
        if gan_key in all_results:
            gan_values = [all_results[gan_key].get(r, np.nan) for r in regions]
            bars1 = ax.bar(x - width/2, gan_values, width, label='GAN',
                          color='steelblue', edgecolor='black', linewidth=0.5)
            
            # Add value labels
            for bar, value in zip(bars1, gan_values):
                if not np.isnan(value):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                           f'{value:.1%}', ha='center', va='bottom', fontsize=10)
        
        # Cokriging data
        if cok_key in all_results:
            cok_values = [all_results[cok_key].get(r, np.nan) for r in regions]
            bars2 = ax.bar(x + width/2, cok_values, width, label='Cokriging',
                          color='lightsteelblue', edgecolor='black', linewidth=0.5)
            
            # Add value labels
            for bar, value in zip(bars2, cok_values):
                if not np.isnan(value):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                           f'{value:.1%}', ha='center', va='bottom', fontsize=10)
        
        ax.set_xlabel('Watershed', fontsize=14)
        ax.set_ylabel('Averaged Drought Area (%)', fontsize=14)
        #ax.set_title(f'SPI{period} - GAN vs Cokriging Comparison', fontsize=16)
        ax.set_xticks(x)
        ax.set_xticklabels(regions_eng, rotation=0, fontsize=12)
        ax.legend(fontsize=12)
        ax.grid(True, alpha=0.3, axis='y')
        ax.set_ylim(0, 0.4)  # Adjust based on your data
        ax.tick_params(direction='in', labelsize=12)
        
        plt.tight_layout()
        
        # Save figure with 600 dpi
        fig_path = os.path.join(output_dir, f'SPI{period}_comparison.png')
        fig.savefig(fig_path, dpi=600, bbox_inches='tight')
        print(f"Saved figure to: {fig_path}")
        
        plt.show()

# 4.2) All SPI periods combined comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, period in enumerate(spi_periods):
    ax = axes[idx]
    gan_key = f'SPI{period}_GAN'
    cok_key = f'SPI{period}_Cok'
    
    x = np.arange(len(regions))
    width = 0.35
    
    # GAN data
    if gan_key in all_results:
        gan_values = [all_results[gan_key].get(r, np.nan) for r in regions]
        bars1 = ax.bar(x - width/2, gan_values, width, label='GAN',
                      color='steelblue', edgecolor='black', linewidth=0.5)
    
    # Cokriging data
    if cok_key in all_results:
        cok_values = [all_results[cok_key].get(r, np.nan) for r in regions]
        bars2 = ax.bar(x + width/2, cok_values, width, label='Cokriging',
                      color='lightsteelblue', edgecolor='black', linewidth=0.5)
    
    ax.set_xlabel('Watershed', fontsize=14)
    ax.set_ylabel('Averaged Drought Area (%)', fontsize=14)
    #ax.set_title(f'SPI{period}', fontsize=16)
    ax.set_xticks(x)
    ax.set_xticklabels(regions_eng, rotation=0, fontsize=12)
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 0.4)  # Adjust based on your data
    ax.tick_params(direction='in', labelsize=12)

plt.suptitle('Comparison of Drought Ratios - GAN vs Cokriging', fontsize=18, y=1.02)
plt.tight_layout()

# Save all SPI periods figure with 600 dpi
combined_fig_path = os.path.join(output_dir, 'all_SPI_comparison.png')
fig.savefig(combined_fig_path, dpi=600, bbox_inches='tight')
print(f"Saved combined figure to: {combined_fig_path}")

plt.show()

# 4.3) Summary statistics comparison
print("\n=== Summary Results - GAN vs Cokriging ===")
for period in spi_periods:
    gan_key = f'SPI{period}_GAN'
    cok_key = f'SPI{period}_Cok'
    
    print(f"\nSPI{period}:")
    print("  GAN:")
    if gan_key in all_results:
        for region, eng_name in zip(regions, regions_eng):
            ratio = all_results[gan_key].get(region, np.nan)
            if not np.isnan(ratio):
                print(f"    {eng_name}: {ratio:.2%}")
            else:
                print(f"    {eng_name}: No data")
        
        # Calculate average
        valid_ratios = [r for r in all_results[gan_key].values() if not np.isnan(r)]
        if valid_ratios:
            avg_ratio = np.mean(valid_ratios)
            print(f"    Average: {avg_ratio:.2%}")
    
    print("  Cokriging:")
    if cok_key in all_results:
        for region, eng_name in zip(regions, regions_eng):
            ratio = all_results[cok_key].get(region, np.nan)
            if not np.isnan(ratio):
                print(f"    {eng_name}: {ratio:.2%}")
            else:
                print(f"    {eng_name}: No data")
        
        # Calculate average
        valid_ratios = [r for r in all_results[cok_key].values() if not np.isnan(r)]
        if valid_ratios:
            avg_ratio = np.mean(valid_ratios)
            print(f"    Average: {avg_ratio:.2%}")

# 4.4) Create a single comprehensive comparison chart
fig, ax = plt.subplots(figsize=(14, 8))

# Prepare data for grouped bar chart
bar_width = 0.15
positions = np.arange(len(regions))
colors_gan = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']  # Blue, Orange, Green, Red
colors_cok = ['#aec7e8', '#ffbb78', '#98df8a', '#ff9896']  # Lighter versions

for i, period in enumerate(spi_periods):
    gan_key = f'SPI{period}_GAN'
    cok_key = f'SPI{period}_Cok'
    
    # Position offset for each SPI period
    offset = (i - 1.5) * bar_width
    
    # GAN bars
    if gan_key in all_results:
        gan_values = [all_results[gan_key].get(r, 0) * 100 for r in regions]  # Convert to percentage
        ax.bar(positions + offset - bar_width/2, gan_values, bar_width,
               label=f'SPI{period}-GAN', color=colors_gan[i], edgecolor='black', linewidth=0.5)
    
    # Cokriging bars  
    if cok_key in all_results:
        cok_values = [all_results[cok_key].get(r, 0) * 100 for r in regions]  # Convert to percentage
        ax.bar(positions + offset + bar_width/2, cok_values, bar_width,
               label=f'SPI{period}-Cok', color=colors_cok[i], edgecolor='black', linewidth=0.5)

ax.set_xlabel('Watershed', fontsize=16)
ax.set_ylabel('Averaged Drought Area (%)', fontsize=16)
#ax.set_title('Comprehensive Drought Analysis - GAN vs Cokriging for All SPI Periods', fontsize=18)
ax.set_xticks(positions)
ax.set_xticklabels(regions_eng, rotation=0, fontsize=14)
ax.legend(ncol=4, loc='upper center', bbox_to_anchor=(0.5, 1.15), fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 40)  # Adjust based on your data
ax.tick_params(direction='in', labelsize=14)

plt.tight_layout()

# Save comprehensive figure with 600 dpi
comprehensive_fig_path = os.path.join(output_dir, 'comprehensive_comparison.png')
fig.savefig(comprehensive_fig_path, dpi=600, bbox_inches='tight')
print(f"Saved comprehensive figure to: {comprehensive_fig_path}")

plt.show()

# Save a summary table of results
results_df = pd.DataFrame()

for period in spi_periods:
    gan_key = f'SPI{period}_GAN'
    cok_key = f'SPI{period}_Cok'
    
    # Create columns for each SPI period and method
    if gan_key in all_results:
        for region in regions:
            results_df.loc[region_display_map.get(region, region), f'SPI{period}_GAN'] = all_results[gan_key].get(region, np.nan)
    
    if cok_key in all_results:
        for region in regions:
            results_df.loc[region_display_map.get(region, region), f'SPI{period}_Cok'] = all_results[cok_key].get(region, np.nan)

# Save to Excel
excel_path = os.path.join(output_dir, 'drought_analysis_results.xlsx')
results_df.to_excel(excel_path)
print(f"Saved summary results to: {excel_path}")

In [ ]:
import os
import matplotlib.pyplot as plt
import geopandas as gpd
import contextily as ctx
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import matplotlib.patches as mpatches

# 파일 경로 설정 - 실제 shapefile 경로로 변경해주세요
watershed_shapefile_path = r'G:\24.11.Graduate/유역경계(grs80).shp'  # 경로를 실제 shapefile 위치로 변경

# 출력 디렉토리 설정
output_dir = 'Figures'
os.makedirs(output_dir, exist_ok=True)

# Shapefile 불러오기
gdf = gpd.read_file(watershed_shapefile_path)

# 좌표계가 WGS84가 아니라면 변환
if gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

# 수역 이름 확인 (GUBUN 컬럼 기준)
regions = gdf['GUBUN'].unique().tolist()
print(f"발견된 수역: {regions}")

# 수역별 색상 매핑
region_colors = {
    '동부수역': '#ff7f0e',  # 주황색
    '서부수역': '#2ca02c',  # 녹색
    '남부수역': '#d62728',  # 빨간색
    '북부수역': '#1f77b4'   # 파란색
}

# 한글 이름과 영어 이름 매핑
region_display_map = {
    '동부수역': 'East',
    '서부수역': 'West', 
    '남부수역': 'South',
    '북부수역': 'North'
}

# 그림 생성
fig, ax = plt.subplots(figsize=(12, 10))

# 배경에 전체 제주도 윤곽 먼저 그리기 (회색으로)
gdf.boundary.plot(ax=ax, color='gray', linewidth=0.5)

# 각 수역별로 다른 색상으로 채우기
for region, color in region_colors.items():
    region_data = gdf[gdf['GUBUN'] == region]
    region_data.plot(ax=ax, color=color, alpha=0.7, edgecolor='black', linewidth=0.5)

# 제목 및 축 설정
#plt.title('Jeju Island Watershed Map', fontsize=16)
ax.set_xlabel('Longitude', fontsize=15)
ax.set_ylabel('Latitude', fontsize=15)

# 범례 생성
legend_patches = []
for region, color in region_colors.items():
    # 영어 이름만 표시
    eng_name = region_display_map.get(region, region)
    label = f"{eng_name}"
    patch = mpatches.Patch(color=color, alpha=0.7, label=label)
    legend_patches.append(patch)

plt.legend(handles=legend_patches, loc='lower right', fontsize=10)

# 축 경계 자동 설정
plt.tight_layout()

# 축 눈금 설정 (보기 좋게)
plt.grid(True, linestyle='--', alpha=0.5)

# 배경 지도 추가 (basemap) - 선택적
try:
    # 참고: contextily를 설치해야 합니다. pip install contextily
    ctx.add_basemap(ax, crs=gdf.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.6)
    print("배경 지도가 성공적으로 추가되었습니다.")
except Exception as e:
    print(f"배경 지도 추가 실패: {e}")
    print("배경 지도 없이 계속합니다.")


# 나침반 추가 (북쪽 방향 표시)
arrow_props = dict(facecolor='black', width=0.1, headwidth=5)
ax.annotate('N', xy=(gdf.total_bounds[0] + 0.05, gdf.total_bounds[3] - 0.05), 
            xytext=(gdf.total_bounds[0] + 0.05, gdf.total_bounds[3] - 0.1),
            arrowprops=arrow_props, ha='center', fontsize=12)

# 정보 텍스트 추가
plt.figtext(0.5, 0.01, '* Areas with SPI <= -1.0 are defined as drought regions', ha='center', fontsize=10, style='italic')

# 저장 및 표시
output_path = os.path.join(output_dir, 'jeju_watershed_map.png')
plt.savefig(output_path, dpi=600, bbox_inches='tight')
print(f"지도가 저장되었습니다: {output_path}")

# 추가: 수역별 정보 테이블 생성
watershed_area = {}
for region in regions:
    region_data = gdf[gdf['GUBUN'] == region]
    area_km2 = region_data.geometry.area.sum() * 111 * 111 / 1000000  # 대략적인 km² 변환 (정확한 투영법 사용 필요)
    watershed_area[region] = area_km2

# 면적 정보 출력
print("\n수역별 대략적인 면적 정보:")
print("=" * 50)
print(f"{'Watershed':<10} {'Area(km²)':<15}")
print("-" * 50)
for region, area in watershed_area.items():
    eng_name = region_display_map.get(region, region)
    print(f"{eng_name:<12} {area:.2f}")
print("=" * 50)

plt.show()